# Bayit Dashboard — Richmond BC Synagogue Site Finder

Interactive map dashboard for identifying properties in Richmond, BC that are zoned to permit Religious Assembly.

**Data sources:**
- ParcelMap BC (128k parcels via WFS)
- City of Richmond Bylaw 8500 (561 assembly-zoned properties)
- OpenStreetMap (parks, schools, hospitals — excluded areas)
- BC ALR boundary
- TransLink GTFS transit stops

## Quick Start
1. Run **Cell 1** (Setup) to install packages
2. Run **Cell 2** (Load Data) to mount Drive and load parquet files
3. Run **Cell 3** (Dashboard) to launch the interactive map

To refresh data from original sources, use the **Data Refresh** section at the bottom.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 1: SETUP — Install packages (run once per session)
# ═══════════════════════════════════════════════════════════════

!pip install -q geopandas lonboard ipywidgets pyarrow requests

import warnings
warnings.filterwarnings('ignore')
print('✓ Packages installed')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 2: LOAD DATA — Mount Google Drive and load parquet files
# ═══════════════════════════════════════════════════════════════

import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path
import os

# Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = Path('/content/drive/MyDrive/bayit-dashboard/data')
    IN_COLAB = True
except ImportError:
    # Local development fallback
    DATA_DIR = Path('data')
    IN_COLAB = False

# Create data directory if it doesn't exist
DATA_DIR.mkdir(parents=True, exist_ok=True)

def load_data():
    """Load all datasets from parquet files."""
    files = {
        'parcels': DATA_DIR / 'parcels.parquet',
        'assembly': DATA_DIR / 'assembly_candidates.parquet',
        'alr': DATA_DIR / 'alr_boundary.parquet',
        'excluded': DATA_DIR / 'excluded_areas.parquet',
        'transit': DATA_DIR / 'transit_stops.parquet',
    }

    missing = [k for k, v in files.items() if not v.exists()]
    if missing:
        print(f'⚠ Missing data files: {missing}')
        print('  Run the "Data Refresh" cells below to download from original sources,')
        print('  or upload parquet files to:', DATA_DIR)
        return None

    data = {}
    for key, path in files.items():
        data[key] = gpd.read_parquet(path)
        print(f'  ✓ {key}: {len(data[key]):,} rows')

    # Enrich assembly candidates with parcel data
    assembly = data['assembly']
    parcels = data['parcels']

    # Join matched assembly candidates with parcel info
    matched = assembly[assembly['matched_parcel_id'].notna()].copy()
    if len(matched) > 0:
        parcel_info = parcels[['id', 'owner_type', 'lot_area_sqm', 'owner_name']].copy()
        parcel_info = parcel_info.rename(columns={'id': 'matched_parcel_id'})
        matched = matched.merge(parcel_info, on='matched_parcel_id', how='left', suffixes=('', '_parcel'))

    unmatched = assembly[assembly['matched_parcel_id'].isna()].copy()
    for col in ['owner_type', 'lot_area_sqm', 'owner_name']:
        if col not in unmatched.columns:
            unmatched[col] = None

    data['assembly_enriched'] = pd.concat([matched, unmatched], ignore_index=True)
    data['assembly_enriched'] = gpd.GeoDataFrame(data['assembly_enriched'], geometry='geom')

    # Compute ALR intersection for assembly parcels with matched polygons
    alr_union = data['alr'].union_all()
    matched_with_geom = data['assembly_enriched'][data['assembly_enriched']['matched_parcel_id'].notna()]
    if len(matched_with_geom) > 0:
        # Get parcel geometries for matched candidates
        parcel_geoms = parcels.set_index('id')[['geom']].rename(columns={'geom': 'parcel_geom'})
        temp = matched_with_geom.merge(
            parcel_geoms,
            left_on='matched_parcel_id',
            right_index=True,
            how='left'
        )
        in_alr = temp['parcel_geom'].apply(
            lambda g: g.intersects(alr_union) if g is not None else False
        )
        data['assembly_enriched'].loc[matched_with_geom.index, 'in_alr'] = in_alr.values
    data['assembly_enriched']['in_alr'] = data['assembly_enriched']['in_alr'].fillna(False)

    print(f'\n✓ All data loaded. {len(data["assembly_enriched"]):,} assembly candidates ({matched.shape[0]} with parcel match).')
    return data

data = load_data()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 3: INTERACTIVE DASHBOARD — Map + Filters
# ═══════════════════════════════════════════════════════════════

import lonboard
from lonboard import Map, PolygonLayer, ScatterplotLayer
from lonboard.colormap import apply_categorical_cmap
import ipywidgets as widgets
from IPython.display import display, clear_output

# ─── Constants ───
RICHMOND_CENTER = (49.166, -123.137)

ZONE_OPTIONS = ['All zones', 'ASY', 'CA', 'CDT1', 'CC', 'CEA']
OWNER_OPTIONS = ['All owners', 'Local Government', 'Private', 'Crown Agency', 'Federal', 'Unclassified']

OCCUPIED_TYPES = {
    'apartments', 'house', 'residential', 'retail', 'restaurant', 'hotel',
    'commercial', 'school', 'cafe', 'supermarket', 'kindergarten',
    'place_of_worship', 'hospital', 'clinic', 'library',
}

# ─── Color helpers ───
def get_parcel_colors(gdf):
    """Color parcels by owner type."""
    colors = np.full((len(gdf), 4), [180, 180, 180, 140], dtype=np.uint8)  # grey default
    colors[gdf['owner_type'] == 'Municipal', :] = [59, 130, 246, 160]      # blue
    colors[gdf['owner_type'] == 'Local Government', :] = [59, 130, 246, 160]
    colors[gdf['owner_type'] == 'Crown Provincial', :] = [139, 92, 246, 160]  # purple
    return colors

# ─── Filter widgets ───
w_zone = widgets.Dropdown(options=ZONE_OPTIONS, value='All zones', description='Zoning:')
w_owner = widgets.Dropdown(options=OWNER_OPTIONS, value='All owners', description='Owner:')
w_min_area = widgets.IntText(value=1000, description='Min area m²:', style={'description_width': 'initial'})
w_max_area = widgets.IntText(value=50000, description='Max area m²:', style={'description_width': 'initial'})
w_exclude_alr = widgets.Checkbox(value=True, description='Exclude ALR', style={'description_width': 'initial'})
w_exclude_occupied = widgets.Checkbox(value=False, description='Exclude occupied', style={'description_width': 'initial'})
w_only_matched = widgets.Checkbox(value=False, description='Only confirmed parcels', style={'description_width': 'initial'})
w_show_base = widgets.Checkbox(value=True, description='Show all parcels (base layer)', style={'description_width': 'initial'})

w_status = widgets.HTML(value='<i>Initializing...</i>')

# ─── Build initial map ───
assembly = data['assembly_enriched'].copy()
parcels = data['parcels'].copy()

# Get parcel geometries for matched assembly candidates
assembly_matched_ids = assembly[assembly['matched_parcel_id'].notna()]['matched_parcel_id'].astype(int).tolist()
assembly_parcel_geoms = parcels[parcels['id'].isin(assembly_matched_ids)].copy()

# Initial filtered view
def apply_filters():
    """Apply all filters and return filtered assembly GeoDataFrame."""
    df = assembly.copy()

    # Zoning
    if w_zone.value != 'All zones':
        df = df[df['zoning'].str.contains(w_zone.value, na=False)]

    # Owner type (only for matched)
    if w_owner.value != 'All owners':
        df = df[df['owner_type'] == w_owner.value]

    # Lot area (only for matched with area data)
    if w_min_area.value:
        df = df[~((df['lot_area_sqm'].notna()) & (df['lot_area_sqm'] < w_min_area.value))]
    if w_max_area.value:
        df = df[~((df['lot_area_sqm'].notna()) & (df['lot_area_sqm'] > w_max_area.value))]

    # ALR
    if w_exclude_alr.value:
        df = df[~df['in_alr'].astype(bool)]

    # Occupied
    if w_exclude_occupied.value:
        df = df[~df['place_type'].isin(OCCUPIED_TYPES)]

    # Only matched
    if w_only_matched.value:
        df = df[df['matched_parcel_id'].notna()]

    return df

# Separate matched (polygons) from unmatched (points)
def get_assembly_layers(filtered):
    matched_ids = filtered[filtered['matched_parcel_id'].notna()]['matched_parcel_id'].astype(int).tolist()
    polys = parcels[parcels['id'].isin(matched_ids)].copy()
    points = filtered[filtered['matched_parcel_id'].isna()].copy()
    return polys, points

# ─── Map output ───
map_output = widgets.Output()

def render_map(change=None):
    """Re-render the map with current filters."""
    filtered = apply_filters()
    assembly_polys, assembly_points = get_assembly_layers(filtered)

    layers = []

    # Base parcel layer
    if w_show_base.value and len(parcels) > 0:
        base_layer = PolygonLayer.from_geopandas(
            parcels,
            get_fill_color=get_parcel_colors(parcels),
            get_line_color=[30, 41, 59, 200],
            line_width_min_pixels=0.5,
            auto_highlight=True,
            pickable=True,
        )
        layers.append(base_layer)

    # Assembly polygon layer (gold)
    if len(assembly_polys) > 0:
        assembly_poly_layer = PolygonLayer.from_geopandas(
            assembly_polys,
            get_fill_color=[245, 158, 11, 160],  # amber
            get_line_color=[180, 83, 9, 220],
            line_width_min_pixels=2,
            auto_highlight=True,
            pickable=True,
        )
        layers.append(assembly_poly_layer)

    # Assembly point layer (gold dots)
    if len(assembly_points) > 0:
        assembly_point_layer = ScatterplotLayer.from_geopandas(
            assembly_points,
            get_fill_color=[245, 158, 11, 200],
            get_line_color=[180, 83, 9, 255],
            get_radius=40,
            line_width_min_pixels=2,
            pickable=True,
        )
        layers.append(assembly_point_layer)

    m = Map(
        layers=layers,
        _initial_view_state={
            'latitude': RICHMOND_CENTER[0],
            'longitude': RICHMOND_CENTER[1],
            'zoom': 13,
            'pitch': 0,
        },
        basemap_style=lonboard.basemap.CartoBasemap.Voyager,
    )

    total = len(assembly)
    shown = len(filtered)
    matched_count = len(assembly_polys)
    point_count = len(assembly_points)

    w_status.value = (
        f'<b style="color:#b45309">{shown}</b> of {total} assembly parcels shown '
        f'({matched_count} polygons, {point_count} points) '
        f'| Base layer: {len(parcels):,} parcels'
    )

    with map_output:
        clear_output(wait=True)
        display(m)

# ─── Wire up filter changes ───
for w in [w_zone, w_owner, w_min_area, w_max_area, w_exclude_alr, w_exclude_occupied, w_only_matched, w_show_base]:
    w.observe(render_map, names='value')

# ─── Layout ───
filter_box = widgets.VBox([
    widgets.HTML('<h3 style="color:#92400e; margin:0">🏛 Assembly Parcel Filters</h3>'),
    widgets.HBox([w_zone, w_owner]),
    widgets.HBox([w_min_area, w_max_area]),
    widgets.HBox([w_exclude_alr, w_exclude_occupied, w_only_matched]),
    w_show_base,
    w_status,
], layout=widgets.Layout(padding='10px', border='2px solid #f59e0b', border_radius='8px', margin='0 0 10px 0'))

display(filter_box)
display(map_output)

# Initial render
render_map()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 4: VIEW FILTERED TABLE — Inspect the current selection
# ═══════════════════════════════════════════════════════════════

filtered = apply_filters()
display_cols = ['address', 'zoning', 'owner_type', 'lot_area_sqm', 'place_type', 'place_name', 'in_alr', 'matched_pid']
available_cols = [c for c in display_cols if c in filtered.columns]

print(f'{len(filtered)} assembly parcels match current filters:\n')
filtered[available_cols].sort_values('lot_area_sqm', ascending=False, na_position='last')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 5: EXPORT — Save filtered parcels to CSV
# ═══════════════════════════════════════════════════════════════

filtered = apply_filters()
export_cols = ['address', 'zoning', 'owner_type', 'lot_area_sqm', 'place_type', 'place_name', 'in_alr', 'matched_pid', 'lat', 'lng']
available_cols = [c for c in export_cols if c in filtered.columns]

export_path = DATA_DIR / 'shortlist.csv'
filtered[available_cols].to_csv(export_path, index=False)
print(f'✓ Exported {len(filtered)} parcels to {export_path}')

# Download in Colab
if IN_COLAB:
    from google.colab import files
    files.download(str(export_path))

---

# Data Refresh

Run these cells to update data from original sources. This replaces the parquet files on Google Drive.

**Sources:**
- **Parcels**: ParcelMap BC (BC Data Catalogue WFS) — ~128k parcels
- **Assembly CSV**: "Properties zoned to permit Religious Assembly" spreadsheet
- **ALR**: BC Agricultural Land Reserve boundaries (WFS)
- **Excluded areas**: OpenStreetMap Overpass (parks, schools, hospitals)
- **Transit**: TransLink GTFS stops

In [ ]:
# ═══════════════════════════════════════════════════════════════
# REFRESH CELL A: Download parcels from ParcelMap BC WFS
# Takes ~5-10 minutes (fetches 50 tiles to stay under 10k WFS cap)
# ═══════════════════════════════════════════════════════════════

import requests
import time
from shapely.geometry import shape

WFS_URL = 'https://openmaps.gov.bc.ca/geo/pub/ows'
LAYER = 'pub:WHSE_CADASTRE.PMBC_PARCEL_FABRIC_POLY_SVW'
MUNICIPALITY = 'Richmond, City of'
BBOX = {'xmin': -123.30, 'ymin': 49.08, 'xmax': -123.00, 'ymax': 49.23}
GRID_COLS, GRID_ROWS = 10, 5

def fetch_parcels_wfs():
    """Fetch all Richmond parcels from BC WFS using grid tiles."""
    dx = (BBOX['xmax'] - BBOX['xmin']) / GRID_COLS
    dy = (BBOX['ymax'] - BBOX['ymin']) / GRID_ROWS

    all_features = []
    total_tiles = GRID_COLS * GRID_ROWS

    for row in range(GRID_ROWS):
        for col in range(GRID_COLS):
            tile_num = row * GRID_COLS + col + 1
            xmin = BBOX['xmin'] + col * dx
            ymin = BBOX['ymin'] + row * dy
            xmax = xmin + dx
            ymax = ymin + dy

            cql = f"MUNICIPALITY='{MUNICIPALITY}' AND BBOX(SHAPE,{xmin},{ymin},{xmax},{ymax},'EPSG:4326')"

            params = {
                'service': 'WFS',
                'version': '2.0.0',
                'request': 'GetFeature',
                'typeName': LAYER,
                'outputFormat': 'application/json',
                'srsName': 'EPSG:4326',
                'CQL_FILTER': cql,
                'count': 10000,
            }

            try:
                resp = requests.get(WFS_URL, params=params, timeout=60)
                resp.raise_for_status()
                features = resp.json().get('features', [])
                all_features.extend(features)
                print(f'  Tile {tile_num}/{total_tiles}: {len(features)} features (total: {len(all_features)})', end='\r')
            except Exception as e:
                print(f'  Tile {tile_num}/{total_tiles}: ERROR - {e}')

            time.sleep(0.5)  # Be polite

    print(f'\n✓ Fetched {len(all_features)} features from {total_tiles} tiles')

    # Convert to GeoDataFrame
    rows = []
    seen_pids = set()
    for f in all_features:
        props = f.get('properties', {})
        pid = props.get('PID')
        if pid in seen_pids:
            continue
        seen_pids.add(pid)

        geom = shape(f['geometry'])
        owner = props.get('OWNER_TYPE', 'Private')
        area = geom.area  # Will compute in m² via geography later

        rows.append({
            'pid': pid,
            'owner_type': owner,
            'lot_area_sqm': props.get('FEATURE_AREA_SQM', 0),
            'civic_address': None,
            'owner_name': None,
            'source': 'parcelmap_bc',
            'geom': geom,
        })

    gdf = gpd.GeoDataFrame(rows, geometry='geom', crs='EPSG:4326')
    gdf['id'] = range(1, len(gdf) + 1)
    print(f'✓ Deduplicated to {len(gdf)} unique parcels')
    return gdf

print('Fetching parcels from ParcelMap BC...')
parcels_fresh = fetch_parcels_wfs()
parcels_fresh.to_parquet(DATA_DIR / 'parcels.parquet')
print(f'✓ Saved to {DATA_DIR / "parcels.parquet"}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# REFRESH CELL B: Geocode assembly CSV addresses
# Takes ~10 minutes (Nominatim rate limit: 1 req/sec)
# ═══════════════════════════════════════════════════════════════

import csv
import re

# Upload the CSV or read from Drive
csv_path = DATA_DIR / 'Properties zoned to permit Religious Assembly.csv'

if not csv_path.exists():
    if IN_COLAB:
        from google.colab import files
        print('Please upload the CSV file:')
        uploaded = files.upload()
        for name, content in uploaded.items():
            (DATA_DIR / name).write_bytes(content)
            csv_path = DATA_DIR / name
    else:
        raise FileNotFoundError(f'CSV not found at {csv_path}')

# Read and deduplicate
with open(csv_path) as f:
    reader = csv.DictReader(f)
    raw_rows = list(reader)

addr_zones = {}
for row in raw_rows:
    addr = row['Address'].strip()
    zoning = row['Zoning'].strip()
    addr_zones.setdefault(addr, set()).add(zoning)

entries = [{'address': a, 'zoning': ', '.join(sorted(z))} for a, z in addr_zones.items()]
print(f'{len(raw_rows)} rows → {len(entries)} unique addresses')

def expand_roads(address):
    variants = [address]
    m = re.search(r'\bNo\s+(\d+)\s+Rd\b', address)
    if m:
        num = m.group(1)
        variants.append(re.sub(r'\bNo\s+\d+\s+Rd\b', f'Number {num} Road', address))
    return variants

def geocode(address):
    """Geocode via Nominatim with road name variants."""
    for variant in expand_roads(address):
        try:
            resp = requests.get(
                'https://nominatim.openstreetmap.org/search',
                params={
                    'street': variant,
                    'city': 'Richmond',
                    'state': 'British Columbia',
                    'country': 'Canada',
                    'format': 'json',
                    'limit': 1,
                    'viewbox': '-123.30,49.08,-123.00,49.23',
                    'bounded': 1,
                },
                headers={'User-Agent': 'BayitDashboard/2.0'},
                timeout=10,
            )
            if resp.ok and resp.json():
                r = resp.json()[0]
                return float(r['lat']), float(r['lon']), r.get('type', ''), r.get('display_name', '').split(',')[0]
        except Exception:
            pass
        time.sleep(1.1)
    return None, None, None, None

results = []
for i, entry in enumerate(entries):
    lat, lng, place_type, place_name = geocode(entry['address'])
    results.append({
        'address': entry['address'],
        'zoning': entry['zoning'],
        'lat': lat,
        'lng': lng,
        'place_type': place_type or '',
        'place_name': place_name or '',
    })
    status = f'({lat:.4f}, {lng:.4f})' if lat else 'FAILED'
    print(f'  [{i+1}/{len(entries)}] {entry["address"]} → {status}', end='\r')
    time.sleep(1.1)

df = pd.DataFrame(results)
geocoded = df[df['lat'].notna()]
print(f'\n✓ Geocoded {len(geocoded)}/{len(df)} addresses')

# Create GeoDataFrame
from shapely.geometry import Point
geoms = [Point(row['lng'], row['lat']) if pd.notna(row['lat']) else None for _, row in df.iterrows()]
assembly_gdf = gpd.GeoDataFrame(df, geometry=geoms, crs='EPSG:4326')
assembly_gdf = assembly_gdf.rename(columns={'geometry': 'geom'}).set_geometry('geom')
assembly_gdf['id'] = range(1, len(assembly_gdf) + 1)
assembly_gdf['matched_parcel_id'] = None
assembly_gdf['matched_pid'] = None
assembly_gdf['notes'] = None

# Spatial match to parcels
if 'parcels_fresh' not in dir():
    parcels_fresh = gpd.read_parquet(DATA_DIR / 'parcels.parquet')

print('Matching to parcels...')
valid_pts = assembly_gdf[assembly_gdf['geom'].notna()].copy()
joined = gpd.sjoin(valid_pts, parcels_fresh[['id', 'pid', 'geom']], how='left', predicate='within')
assembly_gdf.loc[joined.index, 'matched_parcel_id'] = joined['id_right']
assembly_gdf.loc[joined.index, 'matched_pid'] = joined['pid_right'] if 'pid_right' in joined.columns else joined['pid']

matched = assembly_gdf['matched_parcel_id'].notna().sum()
print(f'✓ {matched} of {len(assembly_gdf)} matched to parcel polygons')

assembly_gdf.to_parquet(DATA_DIR / 'assembly_candidates.parquet')
print(f'✓ Saved to {DATA_DIR / "assembly_candidates.parquet"}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# REFRESH CELL C: Download ALR boundary from BC WFS
# ═══════════════════════════════════════════════════════════════

print('Fetching ALR boundary...')
params = {
    'service': 'WFS',
    'version': '2.0.0',
    'request': 'GetFeature',
    'typeName': 'pub:WHSE_LEGAL_ADMIN_BOUNDARIES.OATS_ALR_POLYS',
    'outputFormat': 'application/json',
    'srsName': 'EPSG:4326',
    'CQL_FILTER': "BBOX(SHAPE,-123.30,49.08,-123.00,49.23,'EPSG:4326')",
}
resp = requests.get('https://openmaps.gov.bc.ca/geo/pub/ows', params=params, timeout=60)
resp.raise_for_status()
alr_gdf = gpd.GeoDataFrame.from_features(resp.json()['features'], crs='EPSG:4326')
alr_gdf = alr_gdf.rename(columns={'geometry': 'geom'}).set_geometry('geom')
alr_gdf['id'] = range(1, len(alr_gdf) + 1)
alr_gdf[['id', 'geom']].to_parquet(DATA_DIR / 'alr_boundary.parquet')
print(f'✓ {len(alr_gdf)} ALR polygons saved')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# REFRESH CELL D: Download excluded areas from OSM Overpass
# ═══════════════════════════════════════════════════════════════

from shapely.geometry import shape, Point, mapping

OVERPASS_URL = 'https://overpass-api.de/api/interpreter'
RICHMOND_BBOX = '49.08,-123.30,49.23,-123.00'

query = f"""
[out:json][timeout:120];
(
  way["leisure"="park"]({RICHMOND_BBOX});
  way["leisure"="playground"]({RICHMOND_BBOX});
  way["leisure"="sports_centre"]({RICHMOND_BBOX});
  way["amenity"="school"]({RICHMOND_BBOX});
  way["amenity"="hospital"]({RICHMOND_BBOX});
  way["amenity"="place_of_worship"]({RICHMOND_BBOX});
  way["amenity"="community_centre"]({RICHMOND_BBOX});
  way["amenity"="fire_station"]({RICHMOND_BBOX});
  way["amenity"="police"]({RICHMOND_BBOX});
  way["amenity"="kindergarten"]({RICHMOND_BBOX});
  way["amenity"="childcare"]({RICHMOND_BBOX});
  way["landuse"="cemetery"]({RICHMOND_BBOX});
  relation["leisure"="park"]({RICHMOND_BBOX});
  relation["amenity"="school"]({RICHMOND_BBOX});
  relation["amenity"="hospital"]({RICHMOND_BBOX});
  node["amenity"="school"]({RICHMOND_BBOX});
  node["amenity"="hospital"]({RICHMOND_BBOX});
  node["amenity"="fire_station"]({RICHMOND_BBOX});
  node["amenity"="police"]({RICHMOND_BBOX});
  node["amenity"="kindergarten"]({RICHMOND_BBOX});
  node["amenity"="childcare"]({RICHMOND_BBOX});
);
out body;
>;
out skel qt;
"""

print('Fetching excluded areas from OSM...')
resp = requests.post(OVERPASS_URL, data={'data': query}, timeout=180)
resp.raise_for_status()
osm_data = resp.json()

# Parse nodes and ways into geometries
nodes = {e['id']: (e['lon'], e['lat']) for e in osm_data['elements'] if e['type'] == 'node'}
geometries = []

for el in osm_data['elements']:
    if el['type'] == 'way' and 'nodes' in el:
        coords = [nodes[nid] for nid in el['nodes'] if nid in nodes]
        if len(coords) >= 3:
            from shapely.geometry import Polygon
            try:
                geometries.append(Polygon(coords))
            except Exception:
                pass
    elif el['type'] == 'node' and 'tags' in el:
        geometries.append(Point(el['lon'], el['lat']).buffer(0.0003))  # ~30m buffer

excluded_gdf = gpd.GeoDataFrame(
    {'id': range(1, len(geometries) + 1)},
    geometry=geometries,
    crs='EPSG:4326'
)
excluded_gdf = excluded_gdf.rename(columns={'geometry': 'geom'}).set_geometry('geom')
excluded_gdf.to_parquet(DATA_DIR / 'excluded_areas.parquet')
print(f'✓ {len(excluded_gdf)} excluded areas saved')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# REFRESH CELL E: Download TransLink GTFS transit stops
# ═══════════════════════════════════════════════════════════════

import zipfile
import io

GTFS_URL = 'https://gtfs-static.translink.ca/gtfs/google_transit.zip'

print('Downloading TransLink GTFS...')
resp = requests.get(GTFS_URL, timeout=60)
resp.raise_for_status()

with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
    stops_df = pd.read_csv(z.open('stops.txt'))

# Filter to Richmond area
richmond = stops_df[
    (stops_df['stop_lat'] >= 49.08) & (stops_df['stop_lat'] <= 49.23) &
    (stops_df['stop_lon'] >= -123.30) & (stops_df['stop_lon'] <= -123.00)
].copy()

from shapely.geometry import Point
geoms = [Point(row['stop_lon'], row['stop_lat']) for _, row in richmond.iterrows()]
transit_gdf = gpd.GeoDataFrame(richmond, geometry=geoms, crs='EPSG:4326')
transit_gdf = transit_gdf.rename(columns={
    'stop_name': 'stop_name',
    'stop_lat': 'lat',
    'stop_lon': 'lng',
    'geometry': 'geom',
}).set_geometry('geom')
transit_gdf['id'] = range(1, len(transit_gdf) + 1)
transit_gdf['route_type'] = 3  # bus
transit_gdf[['id', 'stop_name', 'route_type', 'lat', 'lng', 'geom']].to_parquet(DATA_DIR / 'transit_stops.parquet')
print(f'✓ {len(transit_gdf)} transit stops in Richmond saved')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# After refreshing data, re-run Cell 2 (Load Data) and Cell 3
# (Dashboard) to see the updated data on the map.
# ═══════════════════════════════════════════════════════════════

print('Data refresh complete! Re-run Cell 2 and Cell 3 to reload.')